In [ ]:
!pip install pandas numpy torch scikit-learn

In [ ]:
!pip install kaggle

In [ ]:
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
!kaggle datasets download -d ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data
!unzip loghub-hdfs-hadoop-distributed-file-system-data.zip

Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/cli.py", line 68, in main
    out = args.func(**command_args)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 1741, in dataset_download_cli
    with self.build_kaggle_client() as kaggle:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 688, in build_kaggle_client
    username=self.config_values['username'],
             ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'username'
unzip:  cannot find or open loghub-hdfs-hadoop-distributed-file-system-data.zip, loghub-hdfs-hadoop-distributed-file-system-data.zip.zip or loghub-hdfs-hadoop-distributed-file-system-data.zip.ZIP.


In [ ]:
import kagglehub
import os
import pandas as pd

path = kagglehub.dataset_download(
    "ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data"
)

print("Dataset path:", path)

print(os.listdir(path))

100%|██████████| 1.55G/1.55G [00:16<00:00, 103MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3
['HDFS_v2', 'HDFS_2k', 'HDFS_v3_TraceBench', 'HDFS_v1']


In [ ]:
import os

base_path = "/root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3/HDFS_2k"

os.listdir(base_path)

['HDFS_2k.log', 'HDFS_2k.log_templates.csv', 'HDFS_2k.log_structured.csv']

In [ ]:
import pandas as pd
import os

base_path = "/root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3/HDFS_2k"

df = pd.read_csv(os.path.join(base_path, "HDFS_2k.log_structured.csv"))

df.head()

,LineId,Date,Time,Pid,Level,Component,Content,EventId,EventTemplate
0,1,81109,203615,148,INFO,dfs.DataNode$PacketResponder,PacketResponder 1 for block blk_38865049064139...,E10,PacketResponder <*> for block blk_<*> terminating
1,2,81109,203807,222,INFO,dfs.DataNode$PacketResponder,PacketResponder 0 for block blk_-6952295868487...,E10,PacketResponder <*> for block blk_<*> terminating
2,3,81109,204005,35,INFO,dfs.FSNamesystem,BLOCK* NameSystem.addStoredBlock: blockMap upd...,E6,BLOCK* NameSystem.addStoredBlock: blockMap upd...
3,4,81109,204015,308,INFO,dfs.DataNode$PacketResponder,PacketResponder 2 for block blk_82291938032499...,E10,PacketResponder <*> for block blk_<*> terminating
4,5,81109,204106,329,INFO,dfs.DataNode$PacketResponder,PacketResponder 2 for block blk_-6670958622368...,E10,PacketResponder <*> for block blk_<*> terminating


In [ ]:
import re

df["BlockId"] = df["Content"].str.extract(r'(blk_-?\d+)')

sequences = df.groupby("BlockId")["EventId"].apply(list)

sequences.head()

,EventId
BlockId,
blk_-1030832046197982436,[E8]
blk_-1046472716157313227,[E9]
blk_-1049340855430710153,[E10]
blk_-1055254430948037872,[E6]
blk_-1067234447809438340,[E10]


In [ ]:
!git clone https://github.com/wuyifan18/DeepLog.git

Cloning into 'DeepLog'...
remote: Enumerating objects: 189, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 189 (delta 49), reused 48 (delta 47), pack-reused 132 (from 1)
Receiving objects: 100% (189/189), 1.49 MiB | 3.85 MiB/s, done.
Resolving deltas: 100% (97/97), done.


In [ ]:
%cd DeepLog

/content/DeepLog


In [ ]:
!pip install torch pandas numpy scikit-learn

In [ ]:
!cp /root/.cache/kagglehub/datasets/ayenuryrr/loghub-hdfs-hadoop-distributed-file-system-data/versions/3/HDFS_2k/HDFS_2k.log_structured.csv .

In [ ]:
import pandas as pd
import re

df = pd.read_csv("HDFS_2k.log_structured.csv")

df["BlockId"] = df["Content"].str.extract(r'(blk_-?\d+)')

sequences = df.groupby("BlockId")["EventId"].apply(list)

print(sequences.head())

BlockId
blk_-1030832046197982436     [E8]
blk_-1046472716157313227     [E9]
blk_-1049340855430710153    [E10]
blk_-1055254430948037872     [E6]
blk_-1067234447809438340    [E10]
Name: EventId, dtype: object


In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

all_events = [e for seq in sequences for e in seq]
encoder.fit(all_events)

sequences = [[encoder.transform([e])[0] for e in seq] for seq in sequences]

In [ ]:
!ls -R

.:
data			    LICENSE		    README.md
dataView.py		    LogKeyModel_predict.py  requirements.txt
HDFS_2k.log_structured.csv  LogKeyModel_train.py    visual.py

./data:
hdfs_test_abnormal  hdfs_test_normal  hdfs_train


In [ ]:
!python LogKeyModel_train.py

2026-03-15 14:59:00.347471: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-15 14:59:00.353801: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-15 14:59:00.369308: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773586740.398137    4808 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773586740.405361    4808 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773586740.425905    4808 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [ ]:
!ls model

'Adam_batch_size=2048_epoch=30.pt'


In [ ]:
!python LogKeyModel_predict.py

model_path: model/Adam_batch_size=2048_epoch=30.pt
Number of sessions(hdfs_test_normal): 14177
Number of sessions(hdfs_test_abnormal): 4123
elapsed_time: 201.936s
false positive (FP): 1179, false negative (FN): 225, Precision: 76.778%, Recall: 94.543%, F1-measure: 84.739%
Finished Predicting


In [ ]:
normal_sessions = 14177
abnormal_sessions = 4123
FP = 1179
FN = 225

TN = normal_sessions - FP
TP = abnormal_sessions - FN

accuracy = (TP + TN) / (normal_sessions + abnormal_sessions)

print("Accuracy:", accuracy)
print("Accuracy (%):", accuracy * 100)

Accuracy: 0.9232786885245902
Accuracy (%): 92.32786885245902
